# Lab 6: Grounded answers with citations
**Module 3.2: RAG design choices.**  **Time:** 30 minutes.

**Goal:** turn search results into answers that cite their sources, refuse when the answer is not in the
documents, and respect access control. Then measure how chunk size changes retrieval quality.

In [ ]:
# SETUP: run this cell first. It works in Google Colab and in VS Code.
import os, sys, subprocess
REPO_URL = "https://github.com/cybertide-solutions/ai-platform-architect-labs.git"
if "google.colab" in sys.modules:
    if not os.path.exists("/content/course"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/course"], check=True)
    os.chdir("/content/course")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                      # VS Code starts in notebooks/; the course files are one level up
sys.path.insert(0, os.getcwd())
import labkit as lk
lk.setup();
qc, chunks = lk.build_policy_index(size=600, overlap=100)   # the same steps as Lab 5, in one call
print(len(chunks), "chunks indexed")

## 1. Retrieve, then generate
The prompt contains numbered passages. The system prompt tells the model to answer only from them and cite.

In [ ]:
print(lk.RAG_SYSTEM)

In [ ]:
res = lk.rag_answer("What is the hotel limit per night in Mumbai, and what about Indore?", qc)
print(res["answer"])
print("SOURCES:", res["sources"])

What the model actually received (this is what you would log for audit):

In [ ]:
print(lk.format_context(res["hits"])[:1200])

## 2. The "I don't know" path
A grounded assistant must refuse questions the documents do not answer. Otherwise it invents answers.

In [ ]:
for q in ["What is Meridian's revenue this year?", "Can I bring my dog to the office?"]:
    print(q, "\n  ->", lk.rag_answer(q, qc)["answer"], "\n")

## 3. Access control at retrieval time
An HR employee asks about refund approval limits. Their access covers HR documents only.

In [ ]:
q = "Up to what amount can a customer service agent approve a refund?"
print("Customer Service user:", lk.rag_answer(q, qc, dept="Customer Service")["answer"])
print("HR user:              ", lk.rag_answer(q, qc, dept="HR")["answer"])

## 4. Does chunk size matter? Measure it.
For each test question we know which document holds the answer. **Hit rate** = how often that document
is in the top 3 search results.

In [ ]:
tests = lk.load_csv("data/eval_starter.csv")
import pandas as pd

def hit_rate(size, overlap):
    store, ch = lk.build_policy_index(size=size, overlap=overlap)
    hits = [t["source_doc"] in [h["doc"] for h in lk.search(store, t["question"], k=3)] for t in tests]
    return {"chunk_size": size, "overlap": overlap, "chunks": len(ch), "hit_rate": sum(hits) / len(hits)}

pd.DataFrame([hit_rate(300, 50), hit_rate(600, 100), hit_rate(1200, 150)])

## Try this 1: a question that needs two documents (3 minutes)
Losing a laptop is covered by the IT Equipment Policy, and reporting incidents by the Information Security Policy.
Run the cell and look at the sources.

In [ ]:
r = lk.rag_answer("I lost my company laptop while travelling. Who do I tell, and how quickly?", qc)
print(r["answer"])
print("SOURCES:", r["sources"])

**Question:** are both policies cited? If only one is, what would you change: the number of passages (k), the chunk
size, or the prompt?

## Try this 2: only one passage (2 minutes)
The same question as in section 1, but the model now receives only the single best passage (`k=1`).

In [ ]:
r = lk.rag_answer("What is the hotel limit per night in Mumbai, and what about Indore?", qc, k=1)
print(r["answer"])
print("SOURCES:", r["sources"])

**Question:** is the answer still complete? What is the trade-off between a small k (cheaper, faster) and a large k?

## Discussion (end of lab)
Five test questions are far too few to choose a chunk size. How many would you want, and who should write them?
(That is Module 5.)